In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for locating the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates when running from the notebooks folder
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 08. Mathematics of Activation and Loss Functions

> **Learning Objectives**
> - Compare the mathematics, strengths, and weaknesses of Sigmoid, Tanh, ReLU, GELU, and SiLU
> - Understand the formulas and use cases for MSE, cross-entropy, and focal loss
> - Understand why GELU and cross-entropy are standard in LLMs

## 8.1 Why Nonlinear Activations Are Needed

Without activation functions, an MLP is only a composition of linear transformations → another linear transformation. Hidden layers become meaningless.

We need **nonlinearity**. There are several choices.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-6, 6, 200)

# Various activation functions
def sigmoid(x): return 1 / (1 + np.exp(-x))
def tanh(x): return np.tanh(x)
def relu(x): return np.maximum(0, x)
def leaky_relu(x, alpha=0.1): return np.where(x > 0, x, alpha * x)
def gelu(x): return x * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))
def silu(x): return x * sigmoid(x)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fns = [
    (sigmoid, 'Sigmoid'),
    (tanh, 'Tanh'),
    (relu, 'ReLU'),
    (leaky_relu, 'Leaky ReLU'),
    (gelu, 'GELU'),
    (silu, 'SiLU (Swish)'),
]
for ax, (fn, name) in zip(axes.flat, fns):
    ax.plot(x, fn(x), 'b-', linewidth=2)
    ax.set_title(name)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1.5, 3)
plt.tight_layout()
plt.savefig('../figures/ch08_activations.png', dpi=100, bbox_inches='tight')
plt.show()


## 8.2 Sigmoid/Tanh — Saturation and Vanishing Gradients

**Sigmoid**: $\sigma(x) = \frac{1}{1+e^{-x}}$, $\sigma'(x) = \sigma(x)(1-\sigma(x)) \leq 0.25$

Problem: when $|x|$ is large, $\sigma'(x) \to 0$ (saturation). This causes vanishing gradients in deep networks.

**Tanh**: $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$, $\tanh'(x) = 1 - \tanh^2(x) \leq 1$

Better than sigmoid, but still saturates. Its output is near zero-centered.


In [ ]:
# Derivative comparison
def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh_deriv(x):
    return 1 - np.tanh(x)**2

def relu_deriv(x):
    return (x > 0).astype(float)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, sigmoid_deriv(x), label='Sigmoid\' (max=0.25)', linewidth=2)
ax.plot(x, tanh_deriv(x), label='Tanh\' (max=1.0)', linewidth=2)
ax.plot(x, relu_deriv(x), label='ReLU\' (0 or 1)', linewidth=2)
ax.set_title('Activation Function Derivative Comparison — Gradient Flow')
ax.set_xlabel('x'); ax.set_ylabel('f\'(x)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.2)
plt.tight_layout()
plt.savefig('../figures/ch08_derivatives.png', dpi=100, bbox_inches='tight')
plt.show()
print("Sigmoid Derivative Maximum 0.25 → vanishing gradients in deep networks")
print("ReLU derivative is 0 or 1 -> preserves gradient flow")


## 8.3 GELU and SiLU — Why Transformers Use Them

**GELU (Gaussian Error Linear Unit)**: standard in GPT and BERT
$$\mathrm{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}\left[1 + \mathrm{erf}\left(\frac{x}{\sqrt{2}}\right)\right]$$

Approximation: $\mathrm{GELU}(x) \approx 0.5x\left(1 + \tanh\left[\sqrt{2/\pi}(x + 0.044715x^3)\right]\right)$

**SiLU/Swish**: used in LLaMA and related models
$$\mathrm{SiLU}(x) = x \cdot \sigma(x)$$

Both are "smooth" versions of ReLU. For $x < 0$, they are not exactly zero; they keep small negative values.
This improves gradient flow and slightly improves performance.


In [ ]:
# GELU vs ReLU comparison (negative region)
x = np.linspace(-3, 3, 100)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, relu(x), label='ReLU', linewidth=2)
ax.plot(x, gelu(x), label='GELU', linewidth=2)
ax.plot(x, silu(x), label='SiLU', linewidth=2)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('ReLU vs GELU vs SiLU — Differences in the Negative Region')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch08_gelu_vs_relu.png', dpi=100, bbox_inches='tight')
plt.show()

# Values for negative inputs
x_neg = np.array([-0.5, -1.0, -2.0, -3.0])
print("Activation values for negative inputs:")
print(f"{'x':>6} | {'ReLU':>8} | {'GELU':>8} | {'SiLU':>8}")
for xi in x_neg:
    print(f"{xi:>6.1f} | {relu(xi):>8.4f} | {gelu(xi):>8.4f} | {silu(xi):>8.4f}")
print("\nGELU and SiLU allow small negative values in the negative region.")
print("This preserves information better and improves Transformer performance.")


## 8.4 MSE Loss and Regression

**Mean Squared Error (MSE)**: standard for regression problems

$$\mathcal{L}_{\text{MSE}} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$

Derivative: $\frac{\partial \mathcal{L}}{\partial \hat{y}_i} = \frac{2}{N}(\hat{y}_i - y_i)$

Simple, but sensitive to outliers.


In [ ]:
# Visualize MSE loss
def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true)**2)

y_true = 0.0
y_preds = np.linspace(-3, 3, 100)
losses = [mse_loss(yp, y_true) for yp in y_preds]

plt.figure(figsize=(8, 5))
plt.plot(y_preds, losses, 'b-', linewidth=2)
plt.axvline(y_true, color='r', linestyle='--', label=f'Label y={y_true}')
plt.xlabel('Prediction ŷ'); plt.ylabel('MSE Loss')
plt.title('MSE Loss Function (Label y=0)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch08_mse.png', dpi=100, bbox_inches='tight')
plt.show()


## 8.5 Cross-Entropy Loss and Classification

**Binary Cross-Entropy (BCE)**:
$$\mathcal{L}_{\text{BCE}} = -\frac{1}{N}\sum_i \left[y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\right]$$

**Multi-class Cross-Entropy (CE)**: standard for LLMs
$$\mathcal{L}_{\text{CE}} = -\frac{1}{N}\sum_i \log \hat{y}_{i, c_i}$$

Here $c_i$ is the correct class for the $i$-th sample.

## 8.6 The Simple Gradient of Softmax + CE

For softmax $\hat{y} = \mathrm{softmax}(\mathbf{z})$ and CE loss $\mathcal{L} = -\log \hat{y}_c$, the combined gradient is:

$$\frac{\partial \mathcal{L}}{\partial z_i} = \hat{y}_i - \mathbb{1}[i = c]$$

Surprisingly simple! It lowers logits by exactly "prediction - label". It is also numerically stable because it avoids tiny probabilities inside a separate softmax and log.


In [ ]:
# Verify the combined softmax + CE gradient
def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def ce_loss_softmax(z, target_idx):
    """z: logits, target_idx: label class."""
    p = softmax(z)
    return -np.log(p[target_idx] + 1e-12)

# Numerical gradient
def numerical_grad(f, z, h=1e-5):
    grad = np.zeros_like(z)
    for i in range(len(z)):
        z[i] += h
        lp = f()
        z[i] -= 2 * h
        lm = f()
        z[i] += h
        grad[i] = (lp - lm) / (2 * h)
    return grad

# Test
z = np.array([1.0, 2.0, 0.5, -0.5])
target = 1  # label class

# Analytic gradient: p - one_hot(target)
p = softmax(z)
grad_analytic = p.copy()
grad_analytic[target] -= 1

# Numerical gradient
loss_fn = lambda: ce_loss_softmax(z, target)
grad_numeric = numerical_grad(loss_fn, z.copy())

print(f"logits z = {z}")
print(f"softmax p = {p.round(4)}")
print(f"Label = class {target}")
print(f"\nAnalytic gradient (p - one_hot): {grad_analytic.round(4)}")
print(f"Numerical gradient:                  {grad_numeric.round(4)}")
print(f"Error: {np.max(np.abs(grad_analytic - grad_numeric)):.2e}")
print("\n=> The softmax + CE gradient is surprisingly simple: prediction - label")


## 8.7 Label Smoothing

Smooth the target instead of using hard 0/1 labels:
$$y_{\text{smooth}} = (1-\epsilon) \cdot \text{one\_hot} + \frac{\epsilon}{K}$$

- If $\epsilon = 0.1$, the correct class becomes 0.9 and each remaining class gets 0.1/(K-1)
- Prevents overconfidence
- Often used in LLM pretraining


In [ ]:
# Label smoothing
def label_smoothing(target_idx, n_classes, epsilon=0.1):
    """Smooth the labels."""
    smoothed = np.full(n_classes, epsilon / n_classes)
    smoothed[target_idx] += 1 - epsilon
    return smoothed

n_classes = 5
target = 2

hard = np.zeros(n_classes); hard[target] = 1
soft = label_smoothing(target, n_classes, epsilon=0.1)

print(f"Hard label: {hard}")
print(f"Smoothed:   {soft.round(4)}")


## 8.8 Key Takeaways

| Activation | Formula | Strength | Weakness |
|---|---|---|---|
| Sigmoid | $\frac{1}{1+e^{-x}}$ | Smooth | Saturation, vanishing gradients |
| Tanh | $\tanh(x)$ | zero-centered | Saturation |
| ReLU | $\max(0, x)$ | Simple, fast | Dead ReLU |
| GELU | $x\Phi(x)$ | Smooth ReLU | Computation cost |
| SiLU | $x\sigma(x)$ | LLaMA standard | Computation cost |

| Loss | Formula | Use |
|---|---|---|
| MSE | $\frac{1}{N}\sum(y-\hat{y})^2$ | Regression |
| BCE | $-[y\log\hat{y}+(1-y)\log(1-\hat{y})]$ | Binary classification |
| CE | $-\log\hat{y}_c$ | Multi-class classification, LLMs |
| Softmax+CE | $\hat{y} - \text{one\_hot}$ | Simple gradient |

## Exercises

1. Derive directly that the derivative of sigmoid is $\sigma(x)(1-\sigma(x))$.
2. Demonstrate the "dead ReLU" problem experimentally (large negative weights → a neuron stays at 0 forever).
3. Experiment with how GELU vs ReLU affects training in a deep MLP.
4. Compare the gradients of MSE and CE, and explain why CE is more suitable for classification.
5. Compare MNIST MLP accuracy with label smoothing $\epsilon = 0, 0.1, 0.3$.

> Solutions: `solutions/ch08_solutions.ipynb`
